# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aruu28249-boop/Flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My baseline rule

I will create a simple content-prioritization score using observed Google Search Console performance signals from March 2026. The rule will prioritize content pages that show search demand but may have an opportunity for improvement. The score will use search impressions and average search position as the main signals. Pages with higher impressions and weaker average positions will receive a higher priority score because they may have meaningful search visibility but still have room for improvement. The rule is intended to support human review and content-refresh decisions, not to prove that a refresh will improve performance.

### Reason codes

- `HIGH_DEMAND_OPPORTUNITY` — The page has meaningful search impressions and a weaker average position, suggesting an opportunity for review.
- `LOW_DEMAND` — The page has very few search impressions, so there is limited observed search demand to prioritize.
- `STRONG_POSITION` — The page already has a relatively strong average search position, so it is a lower priority for this baseline rule.
- `INSUFFICIENT_DATA` — The page does not have enough usable search performance data to make a confident recommendation.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
import duckdb
from google.colab import userdata

# Create DuckDB connection
con = duckdb.connect()

# Get Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Connect DuckDB to Hugging Face
con.execute(
    f"""
    CREATE OR REPLACE SECRET hf_secret
    (TYPE huggingface, TOKEN '{HF_TOKEN}')
    """
)

# Dataset location
rel = "hf://datasets/FlyRank/internship-warehouse"

print("DuckDB connection is ready!")

DuckDB connection is ready!


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 2: Build the ranked queue
rel = "hf://datasets/FlyRank/internship-warehouse"
# Read March 2026 data
base_query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
"""

df = con.sql(base_query).df()

# Create a simple baseline score
# Higher impressions = more search demand
# Higher average position number = more room for improvement

df["priority_score"] = (
    df["gsc_impressions"].fillna(0) *
    df["gsc_avg_position"].fillna(0)
)

# Assign reason codes
def get_reason(row):
    if row["gsc_impressions"] < 10:
        return "LOW_DEMAND"
    elif row["gsc_avg_position"] >= 10:
        return "HIGH_DEMAND_OPPORTUNITY"
    elif row["gsc_avg_position"] < 10:
        return "STRONG_POSITION"
    else:
        return "INSUFFICIENT_DATA"

df["reason_code"] = df.apply(get_reason, axis=1)

# Assign action labels
df["action"] = df["reason_code"].map({
    "HIGH_DEMAND_OPPORTUNITY": "REVIEW_FOR_REFRESH",
    "LOW_DEMAND": "LOW_PRIORITY",
    "STRONG_POSITION": "MONITOR",
    "INSUFFICIENT_DATA": "NEEDS_DATA_REVIEW"
})

# Rank pages by priority score
df = df.sort_values(
    by="priority_score",
    ascending=False
).reset_index(drop=True)

df["rank"] = df.index + 1

# Select final columns
baseline_queue = df[
    [
        "rank",
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "priority_score",
        "reason_code",
        "action"
    ]
]

# Create output directory
import os
os.makedirs("work/outputs", exist_ok=True)

# Write the ranked queue
output_path = "work/outputs/baseline_action_score.csv"
baseline_queue.to_csv(output_path, index=False)

print("Ranked queue created successfully.")
print("Rows:", len(baseline_queue))
print("Saved to:", output_path)

# Show top 20
baseline_queue.head(20)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Ranked queue created successfully.
Rows: 9841378
Saved to: work/outputs/baseline_action_score.csv


,rank,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,priority_score,reason_code,action
0,1,2026-03-31,client_23a62021009f63c4,content_6530fa9d297c46eb,5364,0,89.848248,481946.0,HIGH_DEMAND_OPPORTUNITY,REVIEW_FOR_REFRESH
1,2,2026-03-31,client_23a62021009f63c4,content_e6df0936699f5b8f,14682,269,25.035826,367576.0,HIGH_DEMAND_OPPORTUNITY,REVIEW_FOR_REFRESH
2,3,2026-03-04,client_62f4a7e64f5e0096,content_945d6ff91386c817,37368,0,8.613948,321886.0,STRONG_POSITION,MONITOR
3,4,2026-03-09,client_23a62021009f63c4,content_36e53e9c707674fc,9409,1,32.640344,307113.0,HIGH_DEMAND_OPPORTUNITY,REVIEW_FOR_REFRESH
4,5,2026-03-11,client_23a62021009f63c4,content_36e53e9c707674fc,9285,16,32.510609,301861.0,HIGH_DEMAND_OPPORTUNITY,REVIEW_FOR_REFRESH
5,6,2026-03-08,client_23a62021009f63c4,content_36e53e9c707674fc,8842,3,33.982809,300476.0,HIGH_DEMAND_OPPORTUNITY,REVIEW_FOR_REFRESH
6,7,2026-03-10,client_23a62021009f63c4,content_36e53e9c707674fc,8963,11,32.476180,291084.0,HIGH_DEMAND_OPPORTUNITY,REVIEW_FOR_REFRESH
7,8,2026-03-12,client_23a62021009f63c4,content_36e53e9c707674fc,8049,12,33.864952,272579.0,HIGH_DEMAND_OPPORTUNITY,REVIEW_FOR_REFRESH
8,9,2026-03-09,client_23a62021009f63c4,content_e8a52cf3d5988c07,15394,45,17.296804,266267.0,HIGH_DEMAND_OPPORTUNITY,REVIEW_FOR_REFRESH
9,10,2026-03-29,client_23a62021009f63c4,content_66288edeb93b7c4f,24577,66,10.794239,265290.0,HIGH_DEMAND_OPPORTUNITY,REVIEW_FOR_REFRESH


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 Review

I reviewed the top 20 rows produced by the baseline action score.

For each row, I will record:
- The recommended action.
- The reason code assigned by the rule.
- A confidence note based only on the observed signals available to the rule.
- What could make the recommendation wrong.

The confidence notes are directional rather than definitive because the baseline rule uses only observed search performance signals and does not include qualitative content review or causal evidence.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.# Section 3: Display the Top 20 rows for review

top20 = baseline_queue.head(20).copy()

print("Top 20 baseline recommendations:")
display(top20)


Top 20 baseline recommendations:


,rank,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,priority_score,reason_code,action
0,1,2026-03-31,client_23a62021009f63c4,content_6530fa9d297c46eb,5364,0,89.848248,481946.0,HIGH_DEMAND_OPPORTUNITY,REVIEW_FOR_REFRESH
1,2,2026-03-31,client_23a62021009f63c4,content_e6df0936699f5b8f,14682,269,25.035826,367576.0,HIGH_DEMAND_OPPORTUNITY,REVIEW_FOR_REFRESH
2,3,2026-03-04,client_62f4a7e64f5e0096,content_945d6ff91386c817,37368,0,8.613948,321886.0,STRONG_POSITION,MONITOR
3,4,2026-03-09,client_23a62021009f63c4,content_36e53e9c707674fc,9409,1,32.640344,307113.0,HIGH_DEMAND_OPPORTUNITY,REVIEW_FOR_REFRESH
4,5,2026-03-11,client_23a62021009f63c4,content_36e53e9c707674fc,9285,16,32.510609,301861.0,HIGH_DEMAND_OPPORTUNITY,REVIEW_FOR_REFRESH
5,6,2026-03-08,client_23a62021009f63c4,content_36e53e9c707674fc,8842,3,33.982809,300476.0,HIGH_DEMAND_OPPORTUNITY,REVIEW_FOR_REFRESH
6,7,2026-03-10,client_23a62021009f63c4,content_36e53e9c707674fc,8963,11,32.476180,291084.0,HIGH_DEMAND_OPPORTUNITY,REVIEW_FOR_REFRESH
7,8,2026-03-12,client_23a62021009f63c4,content_36e53e9c707674fc,8049,12,33.864952,272579.0,HIGH_DEMAND_OPPORTUNITY,REVIEW_FOR_REFRESH
8,9,2026-03-09,client_23a62021009f63c4,content_e8a52cf3d5988c07,15394,45,17.296804,266267.0,HIGH_DEMAND_OPPORTUNITY,REVIEW_FOR_REFRESH
9,10,2026-03-29,client_23a62021009f63c4,content_66288edeb93b7c4f,24577,66,10.794239,265290.0,HIGH_DEMAND_OPPORTUNITY,REVIEW_FOR_REFRESH


### Top-20 Review

| Rank | Action | Reason Code | Confidence Note | What Would Make It Wrong |
|---|---|---|---|---|
| 1 | REVIEW_FOR_REFRESH | HIGH_DEMAND_OPPORTUNITY | High impressions with very weak average position suggests a possible opportunity. | The impressions may come from irrelevant or broad queries, or the page may not be suitable for a refresh. |
| 2 | REVIEW_FOR_REFRESH | HIGH_DEMAND_OPPORTUNITY | High impressions and a relatively weak position suggest potential for improvement. | The page may already be performing well for its intended purpose, or the impressions may not represent valuable search demand. |
| 3 | MONITOR | STRONG_POSITION | High impressions combined with a stronger average position makes immediate refresh less necessary. | The page could still have a CTR or content-quality problem that this rule does not detect. |
| 4 | REVIEW_FOR_REFRESH | HIGH_DEMAND_OPPORTUNITY | High impressions with a weak average position indicate possible search visibility opportunity. | The page may target queries that are not relevant enough to justify a refresh. |
| 5 | REVIEW_FOR_REFRESH | HIGH_DEMAND_OPPORTUNITY | High impressions and weak position suggest the page may benefit from human review. | The observed daily performance may be temporary or influenced by factors not captured by the rule. |
| 6 | REVIEW_FOR_REFRESH | HIGH_DEMAND_OPPORTUNITY | Strong search demand with a weak position creates a possible improvement opportunity. | A refresh may not improve rankings if the issue is competition or search intent mismatch. |
| 7 | REVIEW_FOR_REFRESH | HIGH_DEMAND_OPPORTUNITY | High impressions and weak average position make this page worth reviewing. | The page may have a technical or external ranking issue that content changes cannot solve. |
| 8 | REVIEW_FOR_REFRESH | HIGH_DEMAND_OPPORTUNITY | Search demand combined with weak position suggests a potential content opportunity. | The page may not have enough relevant demand despite its high impression count. |
| 9 | REVIEW_FOR_REFRESH | HIGH_DEMAND_OPPORTUNITY | High impressions and a weaker position suggest possible room for improvement. | The page may already satisfy its target audience, and a refresh could provide little benefit. |
| 10 | REVIEW_FOR_REFRESH | HIGH_DEMAND_OPPORTUNITY | Meaningful impressions with a position around the lower end of page-one visibility suggest review may be useful. | The page may already be performing adequately and may not need a refresh. |
| 11 | REVIEW_FOR_REFRESH | HIGH_DEMAND_OPPORTUNITY | High impressions and weak position indicate a possible opportunity for human review. | The impressions may not represent valuable or relevant search traffic. |
| 12 | REVIEW_FOR_REFRESH | HIGH_DEMAND_OPPORTUNITY | High search demand with a weaker position suggests potential for improvement. | The ranking may be caused by factors outside the content itself. |
| 13 | REVIEW_FOR_REFRESH | HIGH_DEMAND_OPPORTUNITY | The combination of meaningful impressions and weak position makes this page worth investigating. | The daily observation may not represent the page's typical performance. |
| 14 | REVIEW_FOR_REFRESH | HIGH_DEMAND_OPPORTUNITY | Search demand with weak position indicates a possible refresh opportunity. | A refresh may not address the underlying reason for the low ranking. |
| 15 | REVIEW_FOR_REFRESH | HIGH_DEMAND_OPPORTUNITY | High impressions and weak position suggest that further human review could be valuable. | The page may not be strategically important despite its observed search demand. |
| 16 | REVIEW_FOR_REFRESH | HIGH_DEMAND_OPPORTUNITY | The page has observed search demand but a weak average position. | The page may have a temporary ranking fluctuation rather than a persistent problem. |
| 17 | REVIEW_FOR_REFRESH | HIGH_DEMAND_OPPORTUNITY | Meaningful impressions combined with weak position suggest possible improvement potential. | The content may already be optimized, with competition being the main limiting factor. |
| 18 | REVIEW_FOR_REFRESH | HIGH_DEMAND_OPPORTUNITY | High impressions and weak position make this page a candidate for human review. | The impressions may come from low-value or unrelated queries. |
| 19 | REVIEW_FOR_REFRESH | HIGH_DEMAND_OPPORTUNITY | Search demand with weak position suggests a potential opportunity to investigate. | The page may not be suitable for a content refresh or may have a different business priority. |
| 20 | REVIEW_FOR_REFRESH | HIGH_DEMAND_OPPORTUNITY | High impressions and weak position indicate that the page deserves further review. | The observed performance may be an isolated daily result and not a stable pattern. |

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak Picks

The baseline rule has some weaknesses. The most important issue is that the rule ranks daily observations rather than unique content items. This means the same content item can appear multiple times in the Top 20 when it has high scores on several days. This reduces the diversity of the review queue.

Rank 3 is also a weak pick for the `REVIEW_FOR_REFRESH` objective because it has very high impressions and a relatively strong average position of about 8.61. The rule labels it `STRONG_POSITION` and recommends `MONITOR`, which is reasonable, but its very high score shows that multiplying impressions by average position can produce rankings that are difficult to interpret consistently.

Another limitation is that the rule uses only impressions and average position. It does not consider content quality, search intent, click-through rate, seasonality, or whether a page is strategically important. Therefore, the recommendations should be treated as directional decision-support rather than final refresh decisions.

### Leakage Check

The baseline score uses only `gsc_impressions` and `gsc_avg_position` from March 2026 observations. It does not use future performance windows or label-derived columns. It also does not use product flags or future outcome information.

The rule therefore avoids deliberate future-window leakage. However, the baseline should still be improved in future iterations because the daily grain can cause repeated content items to dominate the ranked queue.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4: Leakage check

# Features used by the baseline score
used_features = [
    "gsc_impressions",
    "gsc_avg_position"
]

# Check that no future/label-derived fields are used
future_or_label_fields = [
    "trend_direction",
    "trend_pct",
    "future_clicks",
    "future_impressions",
    "future_position"
]

leaked_fields = [
    field for field in used_features
    if field in future_or_label_fields
]

print("Features used in baseline score:")
print(used_features)

print("\nFuture or label-derived fields detected in score:")
print(leaked_fields)

if len(leaked_fields) == 0:
    print("\nLEAKAGE CHECK: PASSED")
    print("No future-window or label-derived inputs were used in the baseline score.")
else:
    print("\nLEAKAGE CHECK: FAILED")

Features used in baseline score:
['gsc_impressions', 'gsc_avg_position']

Future or label-derived fields detected in score:
[]

LEAKAGE CHECK: PASSED
No future-window or label-derived inputs were used in the baseline score.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.